# Riesgo aproximado: acumulado / días / media de tráfico de los puntos

$$\overline F_{ruta}=\frac{1}{N}\sum_iF_i,\qquad Riesgo_{aprox}(\%)=100\frac{\sum_iDVC_{i,mes}}{d_{mes}\overline F_{ruta}}.$$

El modo **automático** usa los aforos AADF 2017 de la carretera cuando existe su exportación. Si falta esa exportación, la demo usa el **perfil histórico de tráfico por celda incluido en el proyecto**, sin necesitar datos brutos. El origen aparece en la pantalla y en el informe.

El perfil histórico se obtiene con `expm1(log_traffic_mean)`: el notebook original de clustering aplica `log1p` al promedio de tráfico de los años disponibles entre **2008 y 2017**. Es una media de **todas las vías de cada celda**, no un aforo de la A82 en 2017 ni el tráfico observado en mayo. El escenario combina los conteos esperados de 2017 con esta afluencia histórica aproximada. Se usa la media aritmética de los valores de los puntos del mapa, con igual peso por celda.

La referencia comprimida contiene los conteos, las coordenadas y el perfil histórico publicados en el proyecto. Estos valores no son datos sintéticos. No se inventan aforos de 2017 ni se reconstruyen a partir de vehículos-km. Para usar aforos específicos de 2017, prepara `data/route_traffic_2017.csv` desde el AADF original.

Los DVC de una celda pueden pertenecer a otras carreteras. La media de tráfico no identifica vehículos que completan toda la ruta. El cociente aproxima una probabilidad para eventos raros y no está validado a nivel individual. Los huecos quedan sin evaluar. Los días del mismo mes comparten la estimación.

Se necesita `numpy` y `pandas`. Ejecuta las celdas en orden. En `AFLUENCIA` puedes elegir `auto`, `aadf_2017` o `historical_profile`. El modo estricto `aadf_2017` conserva el porcentaje pendiente si falta su fuente o el aforo de algún punto; no sustituye registros contradictorios por medias históricas.


In [ ]:
CARRETERA = "A82"
FECHA = "2017-05-15"
INVERTIR_SENTIDO = False
CARPETA_APP_OVERRIDE = None  # Solo si la aplicación no está en el directorio del notebook o sus carpetas superiores.
AADF_CSV_OVERRIDE = None     # Solo si el AADF original está fuera de esas carpetas.

AFLUENCIA = "auto"  # auto / aadf_2017 / historical_profile


In [ ]:
import base64, gzip, hashlib, io, json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

REFERENCE_TABLES = {'conteos': {'sha256': '1bcafa0016c07637fc133d3384faf6bc2059226ff54b07413dc2948e88362e23', 'gzip_base64': '...'}, 'coordenadas': {'sha256': '683743adde78914e71364592ce23c2412810b2c32f36260651252f7982a10cab', 'gzip_base64': 'H4sIAAAAAAACA22d664ky22l/+tZCoW4B+NphDOG7BEgy4AsDDBv7/WRjNxVW+6Wje7dzMwIBi+Ltzj/9pe//e31lz/++59//ft//Pk///LH319//69//PP/Pn/723/9x5//+Y8//v3f//pv8YP//5c//vHff/7j//3x17/98X/+9pc/jbbGq41eSnmX1+wn/rDerZRd6zm96d+szFctfxp91FerdfcJ8egriY/N2hvPlj6sJW1/td5HvFi0M2jXmae1Wc1q32ckrS9i1vKeop2xiP2ufem3rbPbHrmG0cqrlXlOfTcRT733vPueL3uX03ctw5adaSepY8WxCn3Z/2DvpS1V22eONebpSdterVVL2jHm9FXYbmPrs2MvXp607G4eK+8h2m2jvZd2V3rXinbrta1V910y29PrnW1TjzXtUy/Wzvpq54yiPd8Xz1ebyYE525mxYL2zHK13TdNP74LXS5+xVt/9+cXKy7afnww4U0rV1vaxtuYa+fR+tV1qfmnN+yVO3kTdGztMWns1fWfGDsbKAxpi0tAbxqh7Ja2e1wGxrXevOqHVyt4sx8+odw66LFvwMx9wXrZWvraxJB7bV1QkaPBzlVVaisws8FSi2H891Jdk7PnFfs6RHJ6i09Ah7HyavWsX8x1LXLW+Z2OBYrkWOCTGc2y75Gxf7BvvsqHva/T3glz608vs1uvikSQ/r3a2ZAxurXGSW72Z6FoX+8S45FatL7HpbL1PtIs/+DqQIv1cEqLdr3xxba/eSgrH2pJ2/eG8i82ig52cRB2WtP3Ve60LWRPt2ZydHtDzQ6zQKUoPSyjrNM7MQjvmnvqIr/i0ebSx0dsUi06uwjgvS8HZ5WprQw1MnJyS8XppEee74l1PWoF+MAN7tyV96UmKNEvTzttEKZETB/TMHmu+8+9VRkUy0fXTKraeKmXIhzlQO/mdUdMyzaLXD1gpsXs+xGmiY07bZVqci1Uif45JGKRn/b7Xj1JW5d1l23TEW2I+92paJjpTZRzq4AyOvrLF0V2lYsnWU156bOtpiauekt162yjxq/KUmNm75Lqu0vPgDhKhxw4Ht2X/+MN5r13EMDN9b8LWoEUgZLZlea7I75dLznjvHwtw3mehdej27m2l7B1ERExb/iUZqNC3NqWeS2d59pC8Ou1aGMeextEX4QzWoiaLNZOW1pG0nGTT33mv1VXbe2PwtM+919YqZHX7SWI3Qykhdi3fW1qy9PG2MFmph1IPnca4i8iH7C1zb1hV6ZrMTpBuWK+/+t7E8x2kUqmKtIivOqyVtHBcmhKvhWWf5lPMMxQctZCZq5YPwby1wiRaL/b1kKz7qV12VDIqC1DvqsZLHlIn7g/taVd1dApDsmdbfKxOu93Z6ii/LZw06PyycBIo8a0vk2CcXJ5O84V6L//SaTWVVEcg9S92sMNnJS1KetmK7Kf7bXWgErW4f09ajktmYeD4juSsw+C3RENGRt5MitpS0fZwRZNiyJPfX4sv1FKl5PqEONCr7JDJJkqK8jFO+UqEPrHi6DaWR98WJJH7lUqIdHLKO0HGWTMFoh8shI5fvr+s5OfklEuLUz49aGUQxYmGuWxjtZn+fU8OSnDGT1cwaAVLtO9tXWo4cSmhhOa4QWcqLjzH0l9Y7b7ehi2vhrzIlkuzdj6EEW0l1Ehe5eTKm3PidEnpsJG0+DpLBFXmPcsuQwvuOk1oZrakhedrtnjvmHmWS1pYdaBbBCWNm4RW3JOmf0mY1t2+fajsD75QrxLnd7N8GHa2hIBS6LS4sltFtkYSUmTdL4u6s3Mmcb/4YbYtxyy3J2+HLQva9eprf61h6yGr9ZeGyf4KaFQZBOlnyWUd8Vc2XZBDeKyi7G5OllYlGAnG1NnGIZ/CwaUjE1pbI1GNNMGk8wfT2kfScl4o8HvYD6+qeNhkXD94JaiiH4JuhC3vhzjAZu6CV8VRh90SDtjSQkG7ko4MocKOfVv0BVw+7/Gt9RJulikXLGHPp93yJp+limn1hpC841E81rMjNDnhrkz0pdXKjx0tV2hfS0haVFJSHRuQdw+XKYcvcCVDJtzb8/zEcp11Im2t4RppGdhiMhJturFK2vmSrMjN6J/vLz0kSvn8kb+m70CyNvRpnbgl0hQ0kaTMe34PuBfQFtIyk4+pLWX9bOSipLi2xx7K9psc5dZmQexBawFI5Xhl7g9HKfu94FB7x98cvOqfu9RCzypmkdLlnozDlq/41CGJiizXt4uQRTt6dMruyckTJMTjCIDcSS5UBjUwnqzJke3RE9fPHkdMGdwsLNBFbYKBRScrZRX2SFq32xneCcuvK+sbWykkjE23pHW0f3l17nv/d1okQ+7/A6PJrQoflR4yQrAnhZdn08mn8TgOjE7KKdAsjIfMKg5P7DUZ85QnxzjjrrydG6HadishGCn3upMW2QPxK8K7nJ7sV/8TpHiOTuIoEC+rLIAloH4ZehDH8jCppPGU3MjlHGdfuTtA9oTTdKR8QO6vKFJEVDdxZnPYP2qYe8FQxO9G173tx1pKLeTj5LDkd0fQLsC4PJUw56cFlLm3D0+KBQQGm/hQKsBk5eNhqdbAAkqM5VlkPfI3QFLaKwGT6zWwaj4UEbigto7FHvsm9ZpATwCohHwB6Bfu734LYc1IR8R6Nhi2FEHh9BSLtgB6szgq1DuEp6A9T3itUxwETauWc5fjyEHQ+csH4PnH21yozANJbUPKexfjRioNT08YI4O2wDAyHSZiq0Eb8HCHRVakWeVOPJybG1oFczXgiEgdHZYQ0b7bTU6ADsT/KR+wQhtE63g8zKM+6/wEALiwE+oIUJUUCQeQ104qCkmYgykRPCtHEFsBbtIi14rhquxPvT5g+fff88cf2VsYUFKssF7/z3pycyPW4JE3lip+c7iyXjK8n7IhFC3ew+cCSTyNpBsGSYLeFe7KErhSVMyTDKV2VlN0ZWWJp05/jxvmFBY6JGnvFfG/pFXRv75e2+WavQSwE0HI7Y7EQgVUvYtiYjm6kCKJ/6v2nZZDG7mB3pFKHcyBaAN1iFbPKJrHjov2CEcRZL/l13XwE76cBJ2zOl4W/MOwyM8opJfzXvjnJsP3/EQs6iQ7jATXTAQ9aya7hLrsA0nJprsAyHpLIac/RmLgPoTOSS7JBL08yl7v4WhLnk7KTq6mRPwlYrRnJwaUcqVwb6y+trEF8fMQqkN0rbbw2TxuHZ3grdBme86bvRjbkveeWKTkWqB2llX5kiyu1E3E0hYB4Q1eO4mTRezmv4pGB7RIZ2xSKjhsHbNChw7unLkyh+syDd/QbhCJfkM7lg+CFK9Xhml62hVxXBasGgZEIELfl4MhX7JzE9OdRk8x2f1GipI1bXmKtWLHTlqUC/DvtMJaP0oru92JX8+zA1Rp2hW/lseAuRBuxp7qTSlS013EXQOc9LQfOQlyBVtf7CPFeqI4Nf0D+bV8r9ybzIt04oxid72uLk3xhuIUpKwNQkg89EJfZKEbFj2W0Vrk44JtCoUjAJLGyKyTFxNkXtuSNlKgwYowGZ4aEc40UEkB8Vxa9zMSpDeJTm1c4u2uQmw+RyBSwZ7QzKhJHtJeFC/Wn7PXN6SV9iMNMmGKZYg0Kiexdz7tIaokSe5VFkhskW3gcR0NbtIIthcre2M1hpyB5OYQcMfz7nykciT46s/XW20/JnW/SM8MgcMNYC4tIIieRqOEpklpSqj9f3pavqKRAcn/+ceXJzdkwxXGrnza4ZTAqEBJcEra7DlraZZkd+HBOeVLj04JQn5rCWnXX4G/PJ9OV35SyioAH097UNe/NKpz7rX/0rGO6oMedaTr8tmjuhvM3ZyAzIyUa5GemNiAk7ToWKu/od/EaLyfnxAAd46O1BmSljhF9pLsU4r8tIveBYjF33FmFbasKTweP8r+hVzu6yEklsJgR7adjNfdAipK6vSbfZzxN/skNYeUFv83512Ve7vZrraM0BadjzCuvIacqo3LAJT2XyLVSUL4G+jrcPQigUbZz13v06hxSVOC6gWukAbLD5HikCNqd0/nJRsgzBr7rwoGK+ZBsgrO6Y3cUe6AOIw0ib8XAQ8zKdR+tE9Z0O6RU9DiTftwiydanB12ROaFHIJ2Jjb2tCNk3seFsCv8C5mfIo+gt26EuNz3ShOJv4KWtPwv2TMqBIQzAirJ+h6FgwiOSQn07YvRh6VKdZv8e0kp74Xk2Dj5gb5v4kff1G5I00i/LGkjFZYLT3gvBs5ijVMVNJNbTlqMmuxnizXI2L0z/mzEf8N9yrkLdpPWWbBnhhVlyGXK3Qgibc8Myy93T7rLEkoD5IkP2ZV83EN9BSiCHJLe2oCci4KWLIUel6VbXlJQsKK/y/NpOTvddC+RLi+YAay9gI5WujA+ssiyQLKBzXXijUSJHxUj0NbloNu0ebS3/hN6ywjhwc/zEzan0Nrk8yS/VhKRdi+JEIJ+yT7R6K/ckbRYcEXgS6u1dKPd8wkk4SQ+P9EGVur84FnHQtQUlzvstlc+Xd3EIeFbDFAcg9ayAMUqmzyO/5BQxxQQKMLSqnRY92lM3HAs/eIZ+wBtfNIceJAswd+lDHnlRf9A+JQZgQK3dUBSmWv/pmfaJbjifgXE92S2F2PIRwuBCD4BMtivHJScZSXWkMy+vJCJNRIWnC11tHs+RVJdBe45X2Cs4gxZStkHT7tqxwmPtewXdbOVyn/LuZvMn2SlUtywlKDqBihD0TVDW8mmFx0ruyHnkqTYH/m0UAuZphZuQcEPCVHM7Q3DCLH4WqgmKe6EU4pqZPulwzrTWZMW1ZQKyS17UUHkghXCuRJ3HWWA7ooYkSQQ2hfY0wknyuueqBFL8KN8S7GLHHh4VhLo8jokgc91rT1yM4ooZI5O/JZYSN8NlZV389+IjqKoCQIR+/W++3gkymUUyMmurY1vzuzEL48OZCEFquRQ5NxSeLKyRfTexG8jitLTbh+FWOUZ5MN25kqKgnWZQVmH+1FXVHkqwIUQDEDHGStdcbhKlWaGlaFsRGV332rd7J7MGbhD4SUFt8KK33CLRzyNCJBO7fasjham8FCit0kGHY5FuFzGahHY6ozhMkkpUuI63Xnyac/u6OHxZRs26Um3vIq9yPBtwswrNV71koEg/pKKypweYlRKCtq4AB85yY06vSe+cYF4KA2lVkeWSIfwvqvrrFhIpr5jAyybFRuJfn0bGzfvil0/5UbfkeTxVgbkguiHtJWr+PGUHw5gYM/1grt4V9GDViCHIHK5uDQLFnpNxwEuQR7hPuXJTy30C6nJpG2Cmg+gIs+ng+6CZcMRZDwNUMGkKnr8eFoRwFKE+5ES0EaHw8SD9ufDjlwiOU4KtGUhWda/kziQxNR9ad1MpCXZ14/CCBIr0zap0qj3KtDE8cuZ+YvtSf4qjpZJ2+Shz05MPkjjkpf291kWZHT+jZSSULFQeI9SJR0HwgbS26AtO/7wpn2D7Kl21q7CSCVkgMQs4MPysCXixEP6uYj9WkOG1cPLmjtBmDadjKgkQYUiBkkFu+/Nng+pwL6/+YBMFgg3fnuKQCEGfNtE0c+qvAnkG4TqwGzV25NSJKJ8T1/QiedTWBytwL7KDnRPnM9zN3oAMIzay8Yv5NOR4sfc8RDFSh08Ho5UgkLCid1OWrcTNKns80REnAtWgCJDfA0HWYAmhi0aGWQOz73Jf3bSn4sgvWDj9PQUu6i6ypsc9+hUbEA2Qm8ZXw3Pxk2Zb0+hAr9n/FEi2SjcSCBlyNY9NU/JNUeggBqtBwMjDhxkX8Z8yRbUKO5J74TD9PwZmW8anqXTvuxL8WQf6r8EWK1QBKJiM8p92nNxdkJk6ppP3CJTjNOQatZki2feZAlaipeYEbSHLKpAiaLoqJsQ9kh1enZInH5Dr8K6tbi5aJlKUjRnZyb8rJugl4dfnooeNmzna72MTPuQ03oTS3vJ38pnyHcO3EiAVGjRhuwEWpKC1Bz5eGz1JjJeGfzMEWnmXMO+CFyGW/5NAGJRG5lJ634z0ToHNLNQc0j9KoQkaTGS1puafjX2EMn/S/gms6VjpPJfZrlPex+I3MV7frieA7J4V0eEdRUJsxYjh2L5EKIvW1bwPT+f7NRv148u8M7ITPCBfvngkl9MFr/xpSO33MqDn+kMMYRo4WMvIJyeCWuI4AMeB7usnz/xjDTHQ7W6tuNhlT/tzlKa9FFY59udWO9xHSxYDFaAUeXwFOyufBrx3+RxSyQkjkudduEaR05BOE0+QcqbcZFETg5SumS/TsX2r14J2W2v04PSLVoZpvdM2AgDT974dgxJqim84e3l7kNYluer0iSTmtwze80IwAWqKQMk3F3e3tcyJV/2rXYPeqM8hCULZUnrGd6s6ZS+sv4gzaCtY3oLWxrO5bmtcteg07i9cdJW0rPS5J5x8fLElkIGxTK+uWHR6DgxGQNTtG54uTyN1c/dmxBxVPLlxrd8Axw75b7Xa7kr1ys23OreIZFbSNGfy2BPUCFiH3mXwWqEgD9TMYdPyTgI2RPVjXza29vWdx5DtqRlpCzGS3/oZ/HoIx+KLNVlfcmSsD4oOZJsanN951YiJ0XXR3Ee2RpvsVoOutP4J+tEzm0nsQu3YqN3/VyN2PkrUMP0ky4gtizJCM9JCWaN91PpGDytwMi+Sx3ykjRfii+DZEQ+jpiTPXg/Kf/KgsEbp+fP3C3JBAiG6gC6QsmVT3v1UUx6k+QoZHfe56fdS7pjFLEHOnmh2PJ01bArlm2EWNIaSmXJKCK2y0cwX7HfB0Vf6Kdt1NOT7IhMESirBW+250wyFb6J7RJ3dMEcwma54JIeeXseJEvLMkTzlpYNN1DliuhmSSi6PZGxs6Rex+3CcdssKfMe1zSX2/MGKVmbQlAuQg6LHB1HcFMc27MEgAxPlm7PlHqZzhuCKXMT+6Wj25EVqEQQn7yoQEQE7VotYfhCtoZ6/8zuIFJq9GHqjb4utMTFWVBGzhNTuxWjraT1WEO4+f1oFh+SKK6fn3jWqAgm0PI2wJ35cIiYLLUEnY7GJRtknmOljqFNyeNb5h22B/8LsIHtiN5Cb0LFEBdEf9HZauNu40l+fgR5+imVPmpe8Qs/TUTR6QQQsMrU6UZfe5blpZO3LF/dMW3iK7xO0mLT6Yp9H9GKOfO93PTyMxCmLMHdhacEstVkt6f9pw26SiW+Cgx3poa2RVlhJe3FFRyDp4UFHmfKkkfzdl8r8JdCClaQ+Tcq+6lm20P3eWltXHQldSJXs3BAI+XO4/Xb19r6079LHVIbU0ghTiUbPDrvUh9vcaQPjN5dNlf83AWjaCW61F5bG989fXpqlF/gRqGiHiPQpCp5WfPRGLcJkLKnT4coS09FyG4Vb3sMrpV+y2gbXqD7dAhIRfcChKQ/Dfb2aJyqEdGjYnkA74AXMl3f4TTlDcEI4QD+NZ/2ZNmzznBhAuNj0FmqD83xuOXtofcCG1YYKIkmr+RIQuxrtEfJMNxdeR1gk/ljM5Lhd+R6vOefhiUaEJ3WexRWguQtRJr+s2g/6MMBp/ck9aaxhAa93+4MIcUhS92A2lfDzCPcRRtD48QpCHUvxBmuucHMs8cN4LxXie6uePW4BUPbtDMin0I/GQFYBLnJN/ifSW1tU2ZQ8k1AnKTeutYDw3j5NsCM3I1MDcl4tzRBG7b85O4eW66YhniYhoESbfWijZx0YC5q9eGNOrmDSgKeKsRMWgcol8FrprXQQgXnxNx5YwXz2PIkVuhjPnMj2DNJkile35cJ3sWx77Hds0CVFPdJASVqITrmYSPCofBWPyT5g0QPIVeQz5gbUUta76rOtqderlmpMiBiBOCcen7SemErBaYve4qx2iz2RxHZyFDUog24eGZ0kyKmByN/UzXXQTAFsxSk0ByeD4UYy1X1ftE+J35oNvhxINqOf5FAm/maPHsPKl2SPlW70zf6/XQB2E8jT2YlS0mHphAqu745FCxPjZr+4R1ypDNJ0aAZ1eD9VMjfDKss8gyFcQxLWuR8EoBg/WStl7lC0yOnb3sbAKMVQT1i/mJVf7P8Q2BmUhWlDC+kl2TWiegsjd3A7IRIOpItjSawfjcXrcPpX8ZtYXubd8PQkVpQjqCdkaUMtSBpOtPRNDwHLpsqc9J6WfREBZ8BlsAhVPAV75MCW2miTnQeEMeNdn/xfsXQkownLqTsgu4lzrubnVEoCLUb66eH6ZAC0Gk+NbbjbQudlMJz5pIrcVOLugMNdHpLueRc633KmxIoLssXEdETtwA/FCML0JLMFfIPLC5o4Sm6tBjzgYFay+iyYMLAZHiCNvoMpM5OS+3WeURen8IOQfCIFjfRRlIi37tvN0uhO0V7WrLNLQyCaDEzppCL+GoaubAYIelUoxdtxTPqxIwHeTciFZInitZDOpz2Pj8/IXkgq06yni5Z7xvQ0x55fLf/y7CId797+xUGSkmqxLQwNZJPc3R0XoGDJl1Eb/RS4E7Yh9IEXVC5To8UaLQVFj3xm3XqdZ15s/htHmfI/Qz3YzXPxAOGfnnXx+9mbdk2uVF5Q4OTyUSPHGis9IcCRQLRZcl04mJqJ7ectN6LUunzsh/rpCNs9X1+BnHIS5NilG8hZT3zaaDOOjWacyjFpX2ZFEgVedIC1VIMvFk7TfxemVqeOikaD4hqWXOy7LgjLQ9tZoAFYhV/SaOWYozAsWRU6aAMX+s9EteREmVqx2JEj3yTfoIoYvqcVv6lUyjhFCiuEKsKSZyg9RDjWUM7JzPWLg6IM7HspeWYsqthr2dWpXvyVafuld+VtJwODf96FaCCnh4Kg81xEN3oxROA8j0tn+CMWs5prfrk2XXmlPSqlxaTG17nu0kXsem6Pu2wCn8fzuTkSrxN+qJvyefNySOyk0cPQwVJ61nItAn73MZIFickhDGRO0jaGDDLk97jTvkobmg4Cz28E+au6tFCi/7hTd6huSExwg/aevVPUT9c1ZF6u6+166mN8R6ajyZDKUn60QAt/t8A4MiUyRSRDsFeB23A3wQA+7aRCV2R2WVp6zFl1Usptd/3bh+48w5Vhf/e5MV8S9JyEtld4wFHnhreUT7sUOKMup0e4yRWdLttsvDh9sQZIIk0s5IZDFqvLdz4ym7/PyNIYnCvOruW42ViFmtgckaQ6gkDtldG3vWzkF1xJHRIKzppeTbdIUG2IVKhe1rDJRyLnN+Q8UlazyZfsZCju2kBr+GS86EjwJKWg5wu9aIl2ezztDTgyBHTzq+zm0kbI0j53n3fewDwBJP0tEWoTuOAvHG9a6jziTwJ1UojVVkDpdIxjO9d970l+wsBb2IKOtmzD1C0GPD6s4YcHTkMEhVFbMDtuXPB3jSYmf1I5mZ+xKEwFlBQIBnh/jkdIkObmSoXJhqkZWdYgaR1I3CEPRk9FYvdQntuYHh/JPMlO3s1Rf5RyJNqXw0A7+vwDqnG7CJbDP7KqM357QqJfTOW04K9pXFR7wmmDFp9ZU58itDbysLyd6pKw0c+ZZ5eE0pPW9FtpNXS616BjTh1Bk0nzTIzR2SXQ5fVY9hVKGLnDCk9ZYgwTVyZXloxw32NbdnZ9fhm5JyiE/PK8gAzib1rN9NVpd9yh3xnIRxUEBKR2YpZaZo2sUQSMhKE089bsF7eu9BzcZINkU7d1UcDyN91pJoOiIGxE4O03KgmCUYhoLTV+hpuO9ybZRpje0hUusDhCKOm+BRmHMM4gx8rLYgMpKf9GJGMJLvUfBV7VxrvGao/JMAA0iVX7OhA8tXYlUIFWcuY2CSqQLEVS4mDPakRu2triz31S6IfxigN2dtJ6xbPjDTEyzzB+J7NewCK4ycv1kebiaijzYT+YMu8g6Bp8R7ROm/fAXERPfoMvVGQObkuRx7Zn8+qbqM+bQWFcHszbZW0Pg50abP3fL+9oYBmQor4Fu+d3j+WFp5i47oBSKmUHl1JV7I95tLTOBl+JWWqM5SsaIPUe8KJGCvPjjoDcKUhG9IkoiLvMmpJC7IlISx9+in7Ms8nv+S1NMrtjJFSCQ7hyuHy0uIDjzcH1kxw7aYdPRfjSU0CIqdN56d3iztIFS0tGeMgDmQl8+DrulPddPoZuTCtKsulnrohP+GxI73Xtx2xLkYymQdh/iJpkcN05zSsZoZaPKalaTNKvS7zHAGtwJkGlokx6UK7mWJTGTTLRj3Rht8dX9UrbVJq+J4/WVEhd3peKrmLQ908n/bmSxqaaAWUjV1YxZiJIYagsW4SYeeOybhmNtxaeZIdeiVdUdQe2301lQP5eIqTaGnzHmOf/3f1ptun3Kb25c3QwIZ4s+xg+AZStlzkwRLWSqHdH3DZfnKY0n1hQ+IDpntDSWNI/wp4exoSxB+yF9JKGrGTlZ7BPOZBuN4bLamb9IwErHf3Cf2+16OxTYP3T5cfD83z087g/bGLHlK9FCeZH/IEJpcSfJW0aOew8RHx8rQJlE5vgbbM96yYon/2/9G1QhFBJtTOSf8SQ/RCnVzBQMPgiTRHZVJDr5VhrwlTpucfV/TLkga+wQ3T28xwH8+TJa0L/gnl4wKFXAG3PEg2ac+sdjUqunlIzvoRcMbRd8rIo/ZhQPRyiRF9ol+mi5u35S2XxioHso0iFbWtJPZpATJLEAv3VB/E4bwYPyD4mNfIHCRdNvbL65vDwO/0tDAHOs30vkLQOG4Bp9coD8tnjDRj+lBWRAj8Hqe7KMplOsF6edoUO63cWhXOvyYpJzpuhCPwP/LmAQyv4lDtbef861rzU+p7Dtpjqs28bEumMbLU3n1DA0QMXvc7YMjUEb27W6uQHbCk9U7fbPWQkcy5fTIp3IRBtjtn/GlcRpuicdz6Y3QFfwaDfp3yV0LkuCnhZ73jaZda1FyM9LClPMVFCfPSjnttRaNNge4wmobqXS+aZ5f2BHZi8ulQhqG9Ioo90HoWtWT3Z7c7X9m9ndkwbfUa0rgp4Xnvui7WPNIwzG4buYT9gab12iihiw3uB9EO+qxzaztUxUddjZbWlh1RndKICQnVzH6K9mOqhqRt+ggpq8CVUR2nZJe0rihU2T/qpUjFZ2uil1uYd8OMKwq757hRhhWV3x9lQDzme42PJotyOgl6GZlBISKeNpShhS0xrsaIjKGsic6JIIWRwiStL/FExsQnpzq4L1KnTN3Ruac1UeJ1anrO9o3haRlISTyD3JOEgutEYgcbSW85+WNM7ntH3Pt4RwpjcoenkpbSXxZ8vWvhmatb5CGm35cxkxaNXPvr3gCZ5vEzgUzjCF0UkrI72r52NDhFb7KN9tzRYgxHcJETTjQkIq+kyJwsXXPBPlrrKfIyrWYnlx6xI979/CTTjIipCmZ85OLJbXLXjRyUAs9kkkeTsq9fNXXjrp3Qr+6VGlqyZSpTD+KKinvdBN3aKdykiEjkDMWaaWe2h6BcwAPwZbyImpw9XQHUK/UtCrHM1V1mRTDavu9mIKN3I24hyQEyP0x73U/F+LPlsmb5NYfCrAjhvLePRBsm8RvR6RADAB46ifzTm4u+SHY03+Gl9haE0eNgbn7Lp1Y2go245Rl6eHpHiMa0p9RPC4GXl1uegUem1GGIk5gV3QTH8cvbRGiolxa07UFtPuTBQuq/Tn88miDPhvmmZpkfWKgY9YfYI30iOa2tcyKGrkyk3jiEyySspLVnfmRmIMGkofwlXjM6h6W7Pm2TejOfoENwma7y6s2cUUVY3r7HJUJBW+9RVvqgloSaysK673Wv15O23VovU/5MenBjVUkP6SCCbHTQrgfrbK7lOfQvzaxHEr+/5ELTfkj1MoakWY/auvmYRUtaH7JJaz/nDbxpk6XPjbsz7K437ji7fFgn7zPh4goyuT5KmYkI8y6oRkD6MS2rD3CL2iEzE0UTWUsdDrMRwMoT8xvcB4TPJD7hAAXYK8GhN+0aGHB77HePssU0a2Oa1a8X6B9XA/gdDPQ2mbfD2rmf8N4mxRQ/Yoj+kJGun/HXYcyZG5UO8095m9aKe1UqkVn9cDIxwBdjE+YOuLrMzXzI46/LwHknVoc3qpD7Mfv5gBcdO8Damd1yTPMw7TUoC8+WMWBcvVJjusxmude8MSZA+EomMe8/oDLPoHjr37aQJuEvQ0KTrYwsOdDYeT7t0VvMjKI295IXssbU4vxKtRRDzyH0S1vjMgCf1adzj4a8Vu8GMA5ZgyJ5Xu6lDJXxsc20xLwaFnNyUT4zutFjhIw481CYp18rIyTZixfGzm+/MiaZ+0/PjEV+XLaPazdqn5kLEMR57as9y57pc3qpfTirj5k7VMjAhUy+kvV4CPZFqCWJJPuXZy9fdG65fz13MDKfeJgcZ74nm6L0I4wIl5s48bzEhlYal0sxvpPEccPNlaX103jp8xfc58M5zUvsLTIpgT8VC9qfueQMtToJRu9FMyFMa90BAskU+IWO9Z18jntmLM8PrHcTC3xKQky3RM/X1riR6HsJYFwiF26haxdmH89tVMsVzJ/LX/CJtH0Wy4ZvAAwRnn76llUhgTy9UUBWl0sril9CFCVk0bpuze8WHaOu9TsGqgyiSNFLP5eFOas1vxMN9KH+8sRD6kzdhZL1GHeZPlyeaSBFg8tyqBGR82s+pDCXNm5JKsdp8Rtve35xakBSg5nytdGsQPc3tYp7FP37jho/arJuVG+4qOJ+Cf3DW4UMmf+Bu0nAVZjbyTVnSYv+0ShKz+gToRNit887/DBrjetThFg8jxxPt88AUoqynoG0zQ0dSGEOeRAcvYSTuXRF2gvG94ysDDYNBM0HXUvL9O3xXrUS95aaY4hwvJSIGB7gmruslnk+7GQtRQp94xbaQCoXoJT5QJC8KKhe2vFgMwGTQ2+rz+IlbUygk1llwYLqdK1wqU7lajDXnMxOxmU9c3/3eXrDxa9u6+op8Mp4J4OoPL2Lz78w5veWpBkW1Y0gLJe0goFbD+NKpd+/pE1KM9wX3G47rouQP2EUMN4ryXiRiYjdnv4EzNJ18rgzqvdJC8ctyiJ2ngCfd3YyeyQVI1SSQHi1No3qqfemWK5eEFMYqBBgm0nL3vLT3lF0W/58FIGgPsMC+lD9yoAkPXlfJdMrlTCEzE4JG7Xjdo0SN7Qa9amYg0YJaDAF3SY43HG9hqCaYlTAJNDxxB/fOgMyn2TsLFv9d9yncaX6ZA2UqJHrFcl3UUhSHPmn3WgPuDlj8eMBiFjaxkQvoD9e2xycZVvMKT/AkwEr4hCCnJ1si7sa7qcZUMw7ICZiAn4Yfu9FEnsPWMxaMQSSbT+HQUHPW5EQ7En7cXujCO5c1uZ+QRrbyAmGBuy41+CH9rmsb6Dgx6u1LTICO4bYL219YkXmARgv4sYHy831z/z8qfXCX9qsyYfSlHKiWX/3vJr10t60KF0y5/i1l2Vc2k9nJMv2pFC5+IfpAEZPA5Ft73TvTLO/98foniDSYFR/evRQohv7SM2jmVBmC9E/Ucg5lPNTTQbgapLBKgIsuU8flmVC5dO5+E1tv9DZYGKRRhFGU/MW4h2zryXD2nYCEY43jdSbywEO1xRa0jpCD3vG3GXKCxOgTH75tZyWW4+u1Jm0rT+03GnKJN+5ndPb2xck5V4QO51rO9++4Ep7iwJjN7I1aR3ocMGkLB39eJ3pSikldzNTnaNvPxC7iAmXGJv94g2Go/8eGhgU0lhaXiu6HYbRoOMomgTxRUlMrXk5YCWs2zEwmAZIUlUfY0WT5mRoyjK7t2OkLhvuT09kNWg5IIBzzNHCNVOZevWbCD33WgyPfLlRlh4s2picdnquhmFBUAxXctL3SBzJFdWF6YKVY33bLxVjTN/fO7hYNlsD9vAyuSDiinryzlGxFHh07LZ/m9f/9IBsRK5hfBRaz1i3yuHj5cX99Go11zA+7qnTeq/1KZhzerXpnqt3DT6yKAuDhIztdwU5z+QCGLPiStMIInaMQWXb9xmPQ4qJZoKbyRUmTusTebdHlX7XmzejJImXprqbWpmTQin9XHSTZwxCKY4fPFUetD4plHz4ib39QobNeJpflxa0cblK8kG7yRs/aPWib2d61XEnrV/1vO57+0/ah5Y/+kwYQU5aHyno38NZimsZ/vmeZ/T7JrR5si0lV0Xlcmbq7qznUneAhZhNPnLXSDDRNkOU3z34EK05/JzvczgzyiX0SY+k9VNPZ4ZcXmlajGc18hgt0BxtM56Ee2ifKnjxi70ELewhjTnVJJ0/Nz27s5aYUV4K0hqXvCWpndskTUDr9wpwx1Cu1sOJvG79kA7N4KcWMmwOr0bJJUSIcPK9dw7zTW1fmJx5zZ03Eu5sbzc3KVxkdkulnrABnHP3ZHA3OjpXMIpb4fvTvNAP3Zcbf+XvdXTBkN2KdozyBI3dm7m9f7Jk06dluyHjSOy2cNFGDCnQ96RD4rYd6eVMak8I1hMXhpe8uF77q4fMl5hEk31SV4+tVs++kEpHqWeZaawkz1bp+4oOBq5wJasZfU80Oyoa+R5cotqMEeCq69nuJ6Jb0ki3PL94Xjv5neEU4zfD7jIHFhUOqyeW2Jd/lZuVWogiNycxYEgpIurVFDEj8RpLbOz1c4lcQIVXoIGfLvfYGI3CgyKxP8W9ApmNbNEhU/iPDexIlHAv5MsLu0HMKFOMAXIlu3wK2TqFTy4UAhPx5ritmLm4cW8u5SZbqh3cirGSmDcrRqmfgUPFzH6jBgUh2sA2LzLBAH9cUOLF3Qq5sNnWyboR8Qq35nE7QE9a1tWQEoYfoN45COHNk5RSGlfu2vMAa2PQL1/eLe9vW96pz1JPZrJBjyIuceFIrfQjrawzc+sHGbB5TQKK8uJWjBDBSi/3TF3gauzduMZnBGLlv7Xwcg4hpeSuBne6cIX14Qp3mrn90kwLauPVq+WrP64nlzFnFUaiY+arjVdz9WkQl5//mAZWDIxEj1u8mYhmXD/EaPxDfPyaS6ppDvkg5iLhcc0i+a6beKJ4O300iuq/m7L/AZ0I9Y0UZAAA'}}


def load_reference(name):
    reference = REFERENCE_TABLES[name]
    raw = gzip.decompress(base64.b64decode(reference["gzip_base64"]))
    if hashlib.sha256(raw).hexdigest() != reference["sha256"]:
        raise ValueError("La referencia no coincide con su hash.")
    return pd.read_csv(io.BytesIO(raw))

route_counts = load_reference("conteos")
route_coordinates = load_reference("coordenadas")


In [ ]:
"""AADF mean for each displayed route cell, then an equal-weight route mean.

No raw AADF is bundled. Missing, conflicting or directional records block the
corresponding cell mean rather than silently turning it into zero traffic.
"""
from __future__ import annotations
from pathlib import Path
import hashlib
import json
import math
import pandas as pd
import numpy as np

TRAFFIC_FILE = 'route_traffic_2017.csv'
TRAFFIC_COLUMNS = ['cell','year','road_name','daily_vehicle_mean','aadf_count_points','traffic_status']
RAW_COLUMNS = ['count_point_id','year','region_name','road_name','easting','northing','all_motor_vehicles']


def find_aadf(start: Path) -> Path | None:
    for root in [start.resolve(), *start.resolve().parents]:
        path = root / 'Datos/trafico/aadf_data/dft_traffic_counts_aadf.csv'
        if path.is_file():
            return path
    return None


def aggregate_route_traffic(raw: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    if not set(RAW_COLUMNS).issubset(raw.columns):
        raise ValueError(f'Faltan columnas AADF: {sorted(set(RAW_COLUMNS) - set(raw.columns))}')
    frame=raw.loc[raw.region_name.astype(str).str.strip().eq('Scotland')].copy()
    for col in ['year','easting','northing','all_motor_vehicles','count_point_id']:
        frame[col]=pd.to_numeric(frame[col],errors='coerce')
    frame=frame[frame.year.eq(2017)].copy()
    if frame.empty:
        raise ValueError('No hay aforos AADF de Scotland para 2017.')
    frame['road_name']=frame.road_name.astype('string').str.strip().str.upper()
    xy=np.isfinite(frame.easting)&np.isfinite(frame.northing)&frame.easting.between(0,799999.999)&frame.northing.between(0,1299999.999)
    frame['cell']=pd.Series(pd.NA,index=frame.index,dtype='Int64')
    frame.loc[xy,'cell']=(np.floor(frame.loc[xy,'northing']/10000)*80+np.floor(frame.loc[xy,'easting']/10000)).astype('int64')
    valid_id=np.isfinite(frame.count_point_id)&frame.count_point_id.ge(0)&frame.count_point_id.eq(np.floor(frame.count_point_id))
    good_flow=np.isfinite(frame.all_motor_vehicles)&frame.all_motor_vehicles.ge(0)
    direction_cols=[c for c in ['direction_of_travel','direction'] if c in frame]
    directional=pd.Series(False,index=frame.index)
    for col in direction_cols:
        directional |= frame[col].notna()&frame[col].astype(str).str.strip().ne('')
    frame['blocked']=~valid_id|~good_flow|~xy|frame.road_name.isna()|frame.road_name.eq('')|directional
    fingerprint=RAW_COLUMNS+direction_cols
    duplicates=frame.duplicated(fingerprint,keep='first')
    unique=frame.loc[~duplicates].copy()
    groups=unique.loc[valid_id.loc[unique.index]].groupby('count_point_id',dropna=False).size()
    conflicts=set(groups[groups>1].index)
    unique['blocked'] |= unique.count_point_id.isin(conflicts)
    table=unique.dropna(subset=['cell','road_name']).groupby(['cell','year','road_name'],as_index=False).agg(
        daily_vehicle_mean=('all_motor_vehicles','mean'), aadf_count_points=('count_point_id','nunique'),
        blocked=('blocked','any'))
    table['traffic_status']=np.where(table.blocked,'blocked','valid')
    table.loc[table.blocked,'daily_vehicle_mean']=np.nan
    table=table[TRAFFIC_COLUMNS].sort_values(['road_name','cell']).reset_index(drop=True)
    metadata=dict(source='AADF 2017 Scotland', exact_duplicate_rows=int(duplicates.sum()),
                  conflicting_count_points=len(conflicts), invalid_coordinates_rows=int((~xy).sum()),
                  directional_rows=int(directional.sum()), exported_cell_road_rows=len(table),
                  blocked_cell_road_rows=int(table.traffic_status.eq('blocked').sum()),
                  route_mean_method='Arithmetic mean of the displayed cells; each cell is the mean of unique AADF count points of that road in 2017.',
                  complete_route_trips_observed=False,
                  limitations=['La media por celda se limita a la carretera seleccionada; los DVC de la celda pueden incluir otras vías.',
                               'La media de los puntos aproxima afluencia y no mide cuántos vehículos completan la ruta.',
                               'Una media diaria anual se mantiene igual todos los meses; el volumen mensual estimado es media por días.',
                               'Las filas sin coordenadas válidas no pueden asociarse a una celda; requieren revisión en el original.'])
    return table,metadata


def prepare_route_traffic(app_root: Path, source_path: Path | None = None) -> dict:
    source_path=source_path or find_aadf(app_root)
    if source_path is None:
        return {'status':'missing_original','message':'Falta el AADF original de 2017. El modo automático de la demo puede usar el perfil histórico 2008–2017 incluido; el modo AADF 2017 queda pendiente. Se busca Datos/trafico/aadf_data/dft_traffic_counts_aadf.csv en la aplicación y sus carpetas superiores.'}
    header=pd.read_csv(source_path,nrows=0).columns
    missing=set(RAW_COLUMNS)-set(header)
    if missing:raise ValueError(f'Faltan columnas AADF: {sorted(missing)}')
    columns=RAW_COLUMNS+[c for c in ['direction_of_travel','direction'] if c in header]
    parts=[]
    for chunk in pd.read_csv(source_path,usecols=columns,chunksize=100000):
        keep=chunk.region_name.astype(str).str.strip().eq('Scotland')&pd.to_numeric(chunk.year,errors='coerce').eq(2017)
        if keep.any():parts.append(chunk.loc[keep].copy())
    if not parts:raise ValueError('No hay registros Scotland 2017 en el AADF original.')
    table,metadata=aggregate_route_traffic(pd.concat(parts,ignore_index=True))
    digest=hashlib.sha256()
    with source_path.open('rb') as file:
        for block in iter(lambda:file.read(1024*1024),b''):digest.update(block)
    metadata.update(status='prepared',source_filename=source_path.name,source_sha256=digest.hexdigest())
    output=app_root/'data';output.mkdir(parents=True,exist_ok=True)
    # Build first; atomically replace each completed output. Never changes model predictions.
    temporary=output/(TRAFFIC_FILE+'.tmp')
    table.to_csv(temporary,index=False);temporary.replace(output/TRAFFIC_FILE)
    report=output/'route_traffic_2017_metadata.json'
    temporary=report.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(metadata,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8');temporary.replace(report)
    return metadata


def load_route_traffic(app_root: Path) -> pd.DataFrame:
    path=app_root/'data'/TRAFFIC_FILE
    if not path.is_file():
        prepare_route_traffic(app_root)
    if not path.is_file():
        return pd.DataFrame(columns=TRAFFIC_COLUMNS)
    table=pd.read_csv(path)
    if not set(TRAFFIC_COLUMNS).issubset(table.columns):
        raise ValueError('El CSV de afluencia exportado no tiene el formato esperado.')
    if table.duplicated(['cell','year','road_name']).any():
        raise ValueError('Hay duplicados en el CSV de afluencia; vuelve a exportarlo desde AADF.')
    return table


def historical_daily_mean(value):
    """Invert the documented log1p of the 2008–2017 cell traffic mean.

    This is a cell-wide historical average over available years, including all
    roads; it is not an AADF 2017 count for the selected road.
    """
    try:
        value=float(value)
        if not math.isfinite(value) or value<0:
            return None
        recovered=math.expm1(value)
        return recovered if math.isfinite(recovered) else None
    except (TypeError,ValueError,OverflowError):
        return None


def select_route_traffic(frame: pd.DataFrame, mode: str='auto', aadf_available: bool=False):
    if mode not in {'auto','aadf_2017','historical_profile'}:
        raise ValueError('Origen de afluencia no válido.')
    chosen='historical_profile' if mode=='historical_profile' or (mode=='auto' and not aadf_available) else 'aadf_2017'
    result=frame.copy()
    if chosen=='historical_profile':
        values=result['log_traffic_mean'] if 'log_traffic_mean' in result else pd.Series(None,index=result.index)
        result['daily_vehicle_mean']=values.map(historical_daily_mean)
    else:
        values=result['aadf_daily_vehicle_mean'] if 'aadf_daily_vehicle_mean' in result else pd.Series(None,index=result.index)
        status=result['traffic_status'] if 'traffic_status' in result else pd.Series('missing',index=result.index)
        result['daily_vehicle_mean']=values.where(status.eq('valid'))
    return result,chosen

# Localiza la exportación de aforos de la aplicación, o prepara una nueva desde el original.
if CARPETA_APP_OVERRIDE is not None:
    traffic_root = Path(CARPETA_APP_OVERRIDE).expanduser().resolve()
else:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    traffic_root = next((p for p in starts if (p / "backend/app.py").is_file() and (p / "data/predicciones_test_2017.csv").is_file()), None)
    if traffic_root is None:
        traffic_root = Path.cwd() / "artifacts/afluencia_ruta"
if AADF_CSV_OVERRIDE is not None:
    traffic_metadata = prepare_route_traffic(traffic_root, Path(AADF_CSV_OVERRIDE).expanduser().resolve())
route_traffic = load_route_traffic(traffic_root)
print("Tabla de aforos preparada:", traffic_root / "data" / TRAFFIC_FILE)
if route_traffic.empty:
    print("No hay AADF 2017: el modo automático usará el perfil histórico incluido en la referencia.")

aadf_available = (traffic_root / "data" / TRAFFIC_FILE).is_file()


In [ ]:
"""Experimental DVC risk restricted to observed 10 x 10 km BNG cells.

The synthetic polyline is clipped to the available cells. Unknown stretches
receive no rate and no probability; they never receive an assumed zero rate.
"""
from __future__ import annotations

from datetime import date
import math
from typing import Any

CELL_M = 10_000
GRID_COLUMNS = 80


def cell_for_xy(x: float, y: float) -> int:
    return math.floor(y / CELL_M) * GRID_COLUMNS + math.floor(x / CELL_M)


def segment_cell_lengths(a: tuple[float, float], b: tuple[float, float]) -> list[tuple[int, float]]:
    """Split a straight BNG segment at grid borders; return (cell, km)."""
    length_km = math.hypot(b[0] - a[0], b[1] - a[1]) / 1000
    if length_km == 0:
        return []
    cuts = {0.0, 1.0}
    for start, end in zip(a, b):
        if end == start:
            continue
        first = math.floor(min(start, end) / CELL_M) + 1
        last = math.ceil(max(start, end) / CELL_M)
        for border_index in range(first, last):
            fraction = (border_index * CELL_M - start) / (end - start)
            if 0 < fraction < 1:
                cuts.add(fraction)
    cuts = sorted(cuts)
    pieces = []
    for begin, end in zip(cuts, cuts[1:]):
        midpoint = (begin + end) / 2
        x = a[0] + midpoint * (b[0] - a[0])
        y = a[1] + midpoint * (b[1] - a[1])
        pieces.append((cell_for_xy(x, y), length_km * (end - begin)))
    return pieces



In [ ]:
"""Experimental route-level count / days / common daily-flow scenario.

Daily flow defaults to the arithmetic mean of route-point AADF values; counts cannot identify
vehicles completing the entire route. Cell totals may include other roads.
"""
from __future__ import annotations

import calendar
from datetime import date
import math
from typing import Any



def estimate_route_daily_risk(
    points: list[dict[str, Any]], month: int, travel_date: str | None = None,
    daily_route_vehicles: float | None = None, distance_km: float | None = None,
    traffic_basis: str = "aadf_2017",
) -> dict[str, Any]:
    selected_date = date.fromisoformat(travel_date) if travel_date else date(2017, month, 15)
    if selected_date.year != 2017 or selected_date.month != month:
        raise ValueError('La fecha debe pertenecer al mes seleccionado de 2017.')
    if daily_route_vehicles is not None and (not math.isfinite(daily_route_vehicles) or daily_route_vehicles <= 0):
        raise ValueError('La afluencia diaria de la ruta debe ser un número finito mayor que cero.')
    if distance_km is not None and (not math.isfinite(distance_km) or distance_km <= 0):
        raise ValueError('La distancia debe ser un número finito mayor que cero.')
    if traffic_basis not in {"aadf_2017", "historical_profile"}:
        raise ValueError("Origen de afluencia no válido.")
    historical = traffic_basis == "historical_profile"
    traffic_note = ("Perfil histórico 2008–2017 · todas las vías de cada celda; aproximación para la demo."
                    if historical else "AADF 2017 · aforos de la carretera seleccionada por celda.")
    days = calendar.monthrange(selected_date.year, month)[1]
    result = {
        'status': 'unavailable', 'experimental': True, 'validated_individual_probability': False,
        'scope': 'available_cells_only', 'travel_date': selected_date.isoformat(),
        'reference_year': 2017, 'reference_month': month, 'temporal_resolution': 'monthly',
        'days_in_month': days, 'monthly_expected_dvc': None, 'daily_expected_dvc': None,
        'daily_route_vehicles': daily_route_vehicles, 'traffic_source': 'user_scenario' if daily_route_vehicles is not None else 'missing',
        'monthly_vehicle_passes': None, 'events_per_vehicle_pass': None,
        'traffic_basis': traffic_basis, 'traffic_period': [2008,2017] if historical else [2017,2017],
        'traffic_road_specific': not historical, 'traffic_note': traffic_note,
        'automatic_daily_route_vehicles': None, 'traffic_points_used': 0, 'missing_traffic_cells': [],
        'probability_approx': None, 'full_route_probability': None,
        'available_cells': [], 'missing_count_cells': [], 'duplicate_rows_ignored': 0,
        'coverage_fraction': None, 'unknown_geometry_cells': [], 'geometry_status': 'unavailable',
        'covered_distance_km': None, 'uncovered_distance_km': None,
        'formula': 'A = sum(DVC_esperados_mes_por_celda_unica); tasa = A / dias_mes / vehiculos_diarios_ruta; porcentaje_aproximado = 100 * tasa',
        'assumptions': [
            'La afluencia diaria común se aproxima con la media aritmética de los valores diarios de los puntos de la ruta, sin ponderar por longitud.',
            traffic_note,
            'El perfil histórico se recupera con expm1(log_traffic_mean), inversa del log1p documentado en el notebook de clustering. No reconstruye aforos de 2017.',
            'Este promedio no identifica los vehículos que completan la ruta y no verifica su correspondencia con los incidentes.',
            'Accidentes y tráfico se suponen referidos a los mismos recorridos; esta correspondencia no está verificada.',
            'Los DVC son conteos mensuales esperados del Poisson, no las probabilidades mensuales del Random Forest.',
            'Los conteos pertenecen a celdas de 10 x 10 km y pueden incluir otras carreteras; cada celda se suma una sola vez.',
            'Riesgo y tráfico uniformes dentro del mes; la fecha no aporta información diaria adicional.',
            'Solo se acumulan celdas disponibles; los huecos quedan sin evaluar, no con riesgo cero.',
            'El cociente es una tasa media por paso de vehículo y solo aproxima una probabilidad para eventos raros.',
            'La cobertura geométrica corresponde a la polilínea de la demo, no a la carretera real.',
        ],
    }
    if not points:
        result['reason'] = 'No hay puntos disponibles para acumular DVC.'
        return result
    unique = {}
    for point in points:
        try:
            raw_cell = float(point['cell'])
            if not math.isfinite(raw_cell) or raw_cell != math.floor(raw_cell) or raw_cell < 0:
                raise ValueError()
            cell = int(raw_cell)
        except (KeyError, TypeError, ValueError, OverflowError):
            result['reason'] = 'Hay puntos sin identificador de celda válido; no se puede evitar doble conteo.'
            return result
        try:
            count = float(point['pred_count_final'])
            if not math.isfinite(count) or count < 0:
                count = None
        except (KeyError, TypeError, ValueError, OverflowError):
            count = None
        if cell in unique:
            if unique[cell]['count'] != count:
                result['reason'] = 'Hay conteos contradictorios para una misma celda; el acumulado no está disponible.'
                return result
            result['duplicate_rows_ignored'] += 1
        else:
            unique[cell] = {'count': count, 'point': point}
    traffic_values = []
    for cell, entry in unique.items():
        try:
            flow = float(entry['point']['daily_vehicle_mean'])
            if not math.isfinite(flow) or flow < 0:
                raise ValueError()
        except (KeyError, TypeError, ValueError, OverflowError):
            result['missing_traffic_cells'].append(cell)
        else:
            traffic_values.append(flow)
    result['traffic_points_used'] = len(traffic_values)
    if traffic_values and not result['missing_traffic_cells']:
        result['automatic_daily_route_vehicles'] = math.fsum(traffic_values) / len(traffic_values)
    if daily_route_vehicles is None and result['automatic_daily_route_vehicles'] is not None:
        daily_route_vehicles = result['automatic_daily_route_vehicles']
        result.update(daily_route_vehicles=daily_route_vehicles, traffic_source='mean_route_points')
    known = {cell: entry['count'] for cell, entry in unique.items() if entry['count'] is not None}
    result['available_cells'] = sorted(known)
    result['missing_count_cells'] = sorted(set(unique) - set(known))
    if not known:
        result['reason'] = 'No hay conteos mensuales válidos; no se interpretan los valores ausentes como cero.'
        return result
    monthly = math.fsum(known.values())
    result.update(monthly_expected_dvc=monthly, daily_expected_dvc=monthly / days)

    # Coverage is calculated using count availability, independently of the old vehicle-km proxy.
    locations = []
    try:
        for cell, entry in unique.items():
            point = entry['point']
            x, y = float(point['easting']), float(point['northing'])
            if not (math.isfinite(x) and math.isfinite(y) and 0 <= x < 800000 and 0 <= y < 1300000 and cell_for_xy(x, y) == cell):
                raise ValueError()
            locations.append((x, y))
        pieces = [piece for a, b in zip(locations, locations[1:]) for piece in segment_cell_lengths(a, b)]
        distance = math.fsum(length for _, length in pieces)
        if distance > 0:
            covered = math.fsum(length for cell, length in pieces if cell in known)
            scale = 1 if distance_km is None else distance_km / distance
            result.update(coverage_fraction=covered / distance,
                          covered_distance_km=covered * scale,
                          uncovered_distance_km=(distance - covered) * scale,
                          unknown_geometry_cells=sorted({cell for cell, _ in pieces if cell not in known}),
                          geometry_status='complete_demo' if covered >= distance - 1e-9 else 'partial_demo')
    except (KeyError, TypeError, ValueError, OverflowError):
        pass  # Unknown geometry never turns into verified full-route coverage.

    if daily_route_vehicles is None:
        result.update(status='needs_traffic', reason='Falta afluencia válida en uno o más puntos para el origen seleccionado. No se calcula una media incompleta. Puedes usar el perfil histórico de la demo o preparar los aforos AADF 2017.')
        return result
    if daily_route_vehicles <= 0:
        result.update(status='zero_traffic', reason='La media de tráfico es cero; no se puede dividir entre cero ni estimar la probabilidad.')
        return result
    monthly_passes = days * daily_route_vehicles
    rate = monthly / days / daily_route_vehicles
    if not math.isfinite(rate) or not math.isfinite(monthly_passes):
        result['reason'] = 'El escenario no produce una tasa y un volumen de tráfico finitos.'
        return result
    result.update(monthly_vehicle_passes=monthly_passes, events_per_vehicle_pass=rate)
    if rate > 1:
        result.update(status='incompatible_traffic', reason='La tasa supera un atropello por paso de vehículo; no se muestra como probabilidad. Revisa la afluencia y la correspondencia de los conteos.')
        return result
    result.update(status='scenario_estimated', probability_approx=rate,
                  reason='Estimación media para una pasada usando la media de los puntos de la ruta.' if result['traffic_source'] == 'mean_route_points' else 'Escenario con afluencia alternativa introducida. El resultado solo acumula celdas disponibles y no valida el riesgo de la ruta completa.')
    return result


In [ ]:
selected_date = date.fromisoformat(FECHA)
if selected_date.year != 2017:
    raise ValueError("La referencia del frontend corresponde a 2017.")
route_rows = route_counts.loc[
    route_counts.road_name.astype(str).str.upper().eq(CARRETERA.upper())
    & route_counts.year.eq(selected_date.year) & route_counts.month.eq(selected_date.month)
].merge(route_coordinates, on="cell", how="left", validate="many_to_one").merge(route_traffic.rename(columns={"daily_vehicle_mean": "aadf_daily_vehicle_mean"}), on=["cell", "year", "road_name"], how="left", validate="many_to_one")
route_rows, traffic_basis = select_route_traffic(route_rows, AFLUENCIA, aadf_available)
if route_rows.empty:
    raise ValueError("No hay puntos para la carretera y el mes seleccionados.")
if route_rows[["easting_mean", "northing_mean"]].isna().any().any():
    raise ValueError("Faltan coordenadas para representar esta referencia de ruta.")
xy = route_rows[["easting_mean", "northing_mean"]].to_numpy(float)
centered = xy - xy.mean(axis=0)
if len(route_rows) > 1:
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    route_rows["axis"] = centered @ vt[0]
else:
    route_rows["axis"] = 0.0
route_rows = route_rows.sort_values("axis", ascending=not INVERTIR_SENTIDO)
scenario_points = route_rows.rename(columns={"easting_mean": "easting", "northing_mean": "northing"}).to_dict("records")
scenario = estimate_route_daily_risk(scenario_points, selected_date.month, FECHA, traffic_basis=traffic_basis)
print(f"{CARRETERA} · {FECHA} · {len(scenario['available_cells'])} celdas con conteos")
print("DVC esperados acumulados del mes:", scenario["monthly_expected_dvc"])
print("Días del mes:", scenario["days_in_month"])
print("DVC diarios medios:", scenario["daily_expected_dvc"])
print("Media automática de vehículos diarios:", scenario["daily_route_vehicles"])
print("Riesgo aproximado (%):", None if scenario["probability_approx"] is None else 100 * scenario["probability_approx"])
print(scenario["traffic_note"])
print(scenario.get("reason", ""))
display(route_rows[["cell", "pred_count_final", "daily_vehicle_mean", "aadf_count_points", "traffic_status"]])


In [ ]:
# Comprueba invariantes con la referencia real; no introduce una afluencia ficticia.
reverse_scenario = estimate_route_daily_risk(scenario_points[::-1], selected_date.month, FECHA, traffic_basis=traffic_basis)
other_day = selected_date.replace(day=1 if selected_date.day != 1 else 2).isoformat()
other_day_scenario = estimate_route_daily_risk(scenario_points, selected_date.month, other_day, traffic_basis=traffic_basis)
assert scenario["monthly_expected_dvc"] == reverse_scenario["monthly_expected_dvc"]
assert scenario["probability_approx"] == reverse_scenario["probability_approx"] == other_day_scenario["probability_approx"]
assert scenario["full_route_probability"] is None
if scenario["daily_route_vehicles"] is not None:
    assert np.isclose(scenario["daily_route_vehicles"], route_rows.daily_vehicle_mean.mean())
print("Comprobado: acumulado único, sentido y día del mismo mes; sin validación individual.")


In [ ]:
# Guarda una ejecución nueva sin reemplazar datos del frontend.
output = Path.cwd() / "artifacts/riesgo_diario_ruta" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output.mkdir(parents=True, exist_ok=False)
scenario["road_name"] = CARRETERA.upper()
if (traffic_root / "data" / TRAFFIC_FILE).is_file():
    scenario["traffic_export_sha256"] = hashlib.sha256((traffic_root / "data" / TRAFFIC_FILE).read_bytes()).hexdigest()
scenario["reference_hashes"] = {name: item["sha256"] for name, item in REFERENCE_TABLES.items()}
(output / "escenario_riesgo_diario.json").write_text(json.dumps(scenario, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
route_rows[["cell", "year", "month", "pred_count_final", "daily_vehicle_mean", "aadf_count_points"]].to_csv(output / "conteos_acumulados.csv", index=False)
print("Resultados guardados en:", output)


## Abrir la demo V21

Descomprime el paquete y ejecuta `Roadkill_Abrir_Frontend.ipynb`. La demo muestra las métricas usando la media histórica de las celdas cuando no está disponible el AADF 2017. En «Origen de afluencia» puedes seleccionar la fuente; el modo automático prioriza AADF 2017 si existe su exportación.

Para A82 en mayo, la referencia histórica incluida da unos **6426,45 vehículos diarios medios**, un acumulado de **10,79 DVC esperados** y un riesgo aproximado de **0,00542 %** bajo estos supuestos. La afluencia es histórica por celda, no un aforo medido de A82 en mayo de 2017.

Para usar AADF 2017 específico de la carretera, coloca el original en `Datos/trafico/aadf_data/dft_traffic_counts_aadf.csv` del repositorio, o indica su ruta en `AADF_CSV_OVERRIDE` del launcher. La exportación bloquea duplicados contradictorios, tráfico ausente y formatos direccionales ambiguos. Si la exportación existe pero tiene datos incompletos, el modo automático mantiene AADF 2017 y el porcentaje pendiente; no cambia de fuente silenciosamente.

La cobertura sigue correspondiendo a la polilínea de la demo. La distancia no cambia el acumulado ni la media. Una media de tráfico cero no permite dividir y una tasa superior a 1 no se muestra como probabilidad.
